In [9]:
#Katz_Backoff Model for Quadgram Model

In [10]:
#Threshold 
k=1

In [11]:
#Data Path
path='/home/deepakchalla/Desktop/Desktop/NLP/Lab1/train.parquet'

In [12]:
from read_data import get_data
data=get_data(path,n=10_000)


In [13]:
from collections import defaultdict
def get_counts(data,n=4):
  counts={i:defaultdict(int) for i in range(1,n+1)}
  for i in range(1,n+1):
    for sent in data:
      tokens=["<s>"]*(i-1)+sent.split()+["</s>"]*(i-1)
      for j in range(len(tokens)-i+1):
        counts[i][tuple(tokens[j:j+i])]+=1
  return counts

In [14]:
counts=get_counts(data)

In [15]:
V=len(counts[1].values())

In [16]:
Nc={i:defaultdict(int) for i in range(1,5)}
for i in range(1,5):
  for k in counts[i].values():
    Nc[i][k]+=1

In [17]:
def mle_prob(w, h, order):
    c = counts[order].get(h + (w,), 0)
    dis = c
    if c > 0 and Nc[order].get(c, 0) > 0 and Nc[order].get(c + 1, 0) > 0:
        dis = (c + 1) * Nc[order].get(c + 1) / Nc[order].get(c)
    if order == 1:
        return dis / V
    else:
        return dis / counts[order - 1].get(h, 1)

def pkatz(w, h, order):
    if order == 1:
        return mle_prob(w, h, order)
    c = counts[order].get(h + (w,), 0)
    if c >= k:
        return mle_prob(w, h, order)
    else:
        alpha = get_alpha(h, order)
        h_lower = h[1:]
        return alpha * pkatz(w, h_lower, order - 1)

def get_alpha(h, order):
    seen_words = [k[-1] for k in counts[order].keys() if k[:-1] == h and counts[order][k] > 0]
    num = 1.0
    for w in seen_words:
        num -= mle_prob(w, h, order)
    h_lower = h[1:]
    seen_words_lower = [k[-1] for k in counts[order - 1].keys() if k[:-1] == h_lower and counts[order - 1][k] == 0]
    den = 1.0
    for w in seen_words_lower:
        den -= pkatz(w, h_lower, order - 1)
    return num / den

def predict(sent, order=4):
    tokens = ["<s>"] * (order - 1) + sent.split() + ["</s>"] * (order - 1)
    log_prob = 0
    for i in range(len(tokens) - order + 1):
        history = tuple(tokens[i:i + order - 1])
        word = tokens[i + order - 1]
        log_prob += pkatz(word, history, order)
    return log_prob


In [18]:
from math import log10 as log
def predict(sent,order=4):
  tokens=["<s>"]*(order-1)+sent.split()+["</s>"]*(order-1)
  log_prob=0
  for i in range(len(tokens)-order+1):
    history=tuple(tokens[i:i+order-1])
    word=tokens[i+order-1]
    prob=(pkatz((word),history,order))
    log_prob+=log(prob) if prob>0 else 0
  return log_prob

In [19]:
test_path="/home/deepakchalla/Desktop/Desktop/NLP/Lab1/test.parquet"

In [20]:
test_data=get_data(test_path,n=1_000)

In [21]:
print(predict(test_data[5]))

-25.841225243496464
